# Teste 6 da PoC: Baseline centralizado × STEP-GAN Federado com SMPC (modelo final)

**Questão de pesquisa:** o STEP-GAN treinado de forma federada com agregação segura (SMPC) alcança
desempenho de detecção de fraude **equivalente (não-inferior)** ao mesmo modelo treinado de forma
**centralizada**, em que todos os dados ficam reunidos num único servidor, sem privacidade?

**Configurações comparadas** (mesma arquitetura e hiperparâmetros do Teste 4, mesmos dados, mesmos pesos iniciais):

| Configuração | Papel no experimento |
|---|---|
| **Centralizado** | Limite superior: todos os dados de treino num único modelo (sem privacidade) |
| **Federado + SMPC** | Modelo final (Teste 4): 3 silos, secret sharing aditivo modular no Discriminador |
| Federado + FedAvg em claro | Mesmos modelos locais do SMPC com média simples: isola o custo do SMPC do custo da federação |
| Local isolado | Limite inferior: cada banco treina sozinho, sem colaborar (média dos 3 clientes) |

**Cuidados metodológicos:**
- Split 60/20/20 estratificado; o scaler é ajustado **só no treino** e o limiar é escolhido **só na validação**.
- **Mesmo orçamento de computação:** o centralizado faz `HP_EPOCHS` épocas sobre todo o treino, e cada cliente faz
  `HP_EPOCHS` épocas sobre 1/3 dele, ou seja, o mesmo número total de passos de gradiente.
- **Várias sementes:** GANs têm alta variância, então cada configuração roda com `SEMENTES` diferentes e o
  resultado é reportado como média ± desvio padrão.
- **Teste de não-inferioridade:** o Δ = (Federado+SMPC − Centralizado) é pareado por semente, com IC 95% por
  bootstrap no conjunto de teste. Se o limite inferior do IC do ΔROC-AUC for maior que −`MARGEM_NAO_INFERIORIDADE`,
  o federado é considerado não-inferior.

In [ ]:
# -*- coding: utf-8 -*-
"""
Teste 6 da PoC: Baseline Centralizado do Modelo Final
STEP-GAN Semi-Supervisionado: Centralizado × Federado + SMPC (Teste 4)
"""

import os
import copy
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, precision_recall_curve,
                             roc_curve)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# =====================================================================
# --- 0. CONFIGURAÇÃO DO EXPERIMENTO ---
# =====================================================================
SEMENTES = [42, 43, 44]            # Uma execução completa por semente
NUM_CLIENTES = 3
MARGEM_NAO_INFERIORIDADE = 0.01    # Perda máxima aceitável de ROC-AUC do federado em relação ao centralizado
N_BOOTSTRAP = 300                  # Reamostragens do teste para o IC 95% pareado
PRECISAO_MINIMA = 0.70             # Estratégia de limiar do Teste 4: maior recall com precisão >= 70%

# Hiperparâmetros idênticos ao Teste 4 (modelo final)
HP_EPOCHS = 35
HP_BATCH_SIZE = 512
HP_LR_G = 0.0002
HP_LR_D = 0.0001
HP_LATENT_DIM = 100
HP_WEIGHT_DECAY = 1e-4
PESO_FRAUDE = 15.0
RAZAO_NORMAIS_POR_FRAUDE = 200     # Amostragem do Teste 4: 200 normais para cada fraude

# SMPC idêntico ao Teste 4 (ponto fixo 2^24, campo primo 2^31 - 1)
PRECISION_BITS = 24
SCALING_FACTOR = 2 ** PRECISION_BITS
MODULUS = 2147483647

DATASETS = ['creditcard', 'paysim']
CAMINHOS = {
    'creditcard': [
        '/content/drive/MyDrive/Mestrado Cesar/Base de Dados/creditcard/creditcard.csv',
        'base_de_dados/creditcard.zip',        # Executando de dentro de PoC/
        'PoC/base_de_dados/creditcard.zip',    # Executando da raiz do repositório
    ],
    'paysim': [
        '/content/drive/MyDrive/Mestrado Cesar/Base de Dados/Kaggle/paysim.csv',
    ],
}


def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


sns.set_style("whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[SETUP] Dispositivo PyTorch: {device}")


def sincronizar():
    """Garante que o cronômetro só pare depois que a GPU terminar (CUDA é assíncrono)."""
    if device.type == 'cuda':
        torch.cuda.synchronize()


try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Fora do Google Colab: usando apenas caminhos locais.")

In [ ]:
# =====================================================================
# --- 1. DADOS (mesma engenharia de features do Teste 4) ---
# =====================================================================

def ler_base(nome):
    for caminho in CAMINHOS[nome]:
        if os.path.exists(caminho):
            print(f"Carregando '{caminho}'...")
            return pd.read_csv(caminho)
    return None


def amostrar_como_teste4(df, alvo):
    """Mantém todas as fraudes e 200x mais transações normais (mesma amostragem do Teste 4)."""
    fraudes = df[df[alvo] == 1]
    normais = df[df[alvo] == 0]
    normais = normais.sample(n=min(len(fraudes) * RAZAO_NORMAIS_POR_FRAUDE, len(normais)), random_state=42)
    return pd.concat([normais, fraudes]).sample(frac=1, random_state=42).reset_index(drop=True)


def preparar_creditcard(df):
    df = amostrar_como_teste4(df.dropna(subset=['Class']), 'Class')
    df = df.drop(columns=['Time'], errors='ignore')
    df['amount_log'] = np.log1p(df['Amount'].clip(lower=0))
    return df.drop(columns=['Amount']), 'Class'


def preparar_paysim(df):
    df = amostrar_como_teste4(df.dropna(subset=['isFraud']), 'isFraud')
    df['orig_balance_zero'] = (df['oldbalanceOrg'] == 0).astype(int)
    df['dest_balance_zero'] = (df['oldbalanceDest'] == 0).astype(int)
    for col in ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']:
        df[col + '_log'] = np.log1p(df[col].clip(lower=0))
    df['errorBalanceOrig'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
    df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']
    df = df.drop(columns=['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest'])
    df = df.drop(columns=[c for c in ['nameOrig', 'nameDest', 'isFlaggedFraud'] if c in df.columns])
    return pd.get_dummies(df, columns=['type'], drop_first=True, dtype=float), 'isFraud'


PREPARADORES = {'creditcard': preparar_creditcard, 'paysim': preparar_paysim}


def carregar_dados(nome):
    df = ler_base(nome)
    if df is None:
        return None
    df, alvo = PREPARADORES[nome](df)
    X = df.drop(columns=[alvo]).values.astype(np.float32)
    y = df[alvo].values.astype(np.int64)

    # Split 60/20/20 estratificado (igual ao Teste 4 Credit Card)
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

    # MinMaxScaler(-1, 1) alinhado ao Tanh do Gerador, ajustado SÓ no treino
    scaler = MinMaxScaler(feature_range=(-1, 1))
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    # Silos bancários: fatias contíguas do treino (mesma partição do Teste 4)
    tamanho = len(X_train) // NUM_CLIENTES
    clientes = [(X_train[i * tamanho:(i + 1) * tamanho], y_train[i * tamanho:(i + 1) * tamanho])
                for i in range(NUM_CLIENTES)]
    # O centralizado usa exatamente a união dos dados dos clientes
    X_central = np.concatenate([c[0] for c in clientes])
    y_central = np.concatenate([c[1] for c in clientes])

    print(f"[DADOS] Features: {X_train.shape[1]} | Treino: {len(X_central)} (fraudes: {int(y_central.sum())}) "
          f"| Validação: {len(X_val)} (fraudes: {int(y_val.sum())}) | Teste: {len(X_test)} (fraudes: {int(y_test.sum())})")
    return dict(nome=nome, input_dim=X_train.shape[1], clientes=clientes, X_central=X_central, y_central=y_central,
                X_val=X_val, y_val=y_val, X_test=X_test, y_test=y_test)

In [ ]:
# =====================================================================
# --- 2. STEP-GAN (arquitetura, loss e treino idênticos ao Teste 4) ---
# =====================================================================

class Generator(nn.Module):
    def __init__(self, input_dim, latent_dim=HP_LATENT_DIM):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.2),
            nn.Linear(128, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512), nn.BatchNorm1d(512), nn.LeakyReLU(0.2),
            nn.Linear(512, input_dim), nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


def custom_bce_loss(pred, target, real_labels_flag):
    """BCE com peso condicional: erro em fraude real pesa PESO_FRAUDE vezes mais."""
    loss = nn.BCELoss(reduction='none')(pred, target)
    weights = torch.ones_like(target)
    weights[(target < 0.5) & real_labels_flag] = PESO_FRAUDE
    return (loss * weights).mean()


def treinar_gan(estado_G, estado_D, dados_X, dados_y, input_dim, semente):
    """Treino STEP-GAN do Teste 4. Usado igualmente pelo centralizado e por cada cliente."""
    torch.manual_seed(semente)
    G = Generator(input_dim).to(device)
    D = Discriminator(input_dim).to(device)
    G.load_state_dict(estado_G)
    D.load_state_dict(estado_D)
    G.train()
    D.train()

    opt_G = optim.Adam(G.parameters(), lr=HP_LR_G, betas=(0.5, 0.999), weight_decay=HP_WEIGHT_DECAY)
    opt_D = optim.Adam(D.parameters(), lr=HP_LR_D, betas=(0.5, 0.999), weight_decay=HP_WEIGHT_DECAY)
    dataset = TensorDataset(torch.from_numpy(dados_X), torch.from_numpy(dados_y).float())
    loader = DataLoader(dataset, batch_size=HP_BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(semente))

    sincronizar()
    t0 = time.perf_counter()
    for _ in range(HP_EPOCHS):
        for i, (real_data, real_y) in enumerate(loader):
            real_data, real_y = real_data.to(device), real_y.to(device)
            batch_s = real_data.size(0)
            if batch_s < 2:
                continue  # BatchNorm do Gerador exige pelo menos 2 amostras

            # Label smoothing: normal -> 0.9, fraude -> 0.1; amostras sintéticas -> 0.1
            real_labels = torch.where(real_y.unsqueeze(1) == 0,
                                      torch.tensor(0.9, device=device), torch.tensor(0.1, device=device))
            fake_labels = torch.full((batch_s, 1), 0.1, device=device)

            # --- Discriminador ---
            opt_D.zero_grad()
            loss_D_real = custom_bce_loss(D(real_data), real_labels, torch.ones_like(real_labels, dtype=torch.bool))
            fake_data = G(torch.randn(batch_s, HP_LATENT_DIM, device=device))
            loss_D_fake = custom_bce_loss(D(fake_data.detach()), fake_labels, torch.zeros_like(fake_labels, dtype=torch.bool))
            (loss_D_real + loss_D_fake).backward()
            torch.nn.utils.clip_grad_norm_(D.parameters(), max_norm=1.0)
            opt_D.step()

            # --- Gerador (a cada 2 passos do D) ---
            if i % 2 == 0:
                opt_G.zero_grad()
                loss_G = nn.BCELoss()(D(fake_data), torch.full((batch_s, 1), 0.9, device=device))
                loss_G.backward()
                torch.nn.utils.clip_grad_norm_(G.parameters(), max_norm=1.0)
                opt_G.step()
    sincronizar()
    return copy.deepcopy(D.state_dict()), time.perf_counter() - t0

In [ ]:
# =====================================================================
# --- 3. AGREGAÇÃO: SMPC MODULAR (Teste 4) E FEDAVG EM CLARO ---
# =====================================================================

def float_to_fixed_point(tensor):
    # Arredonda (em vez de truncar) para não introduzir viés sistemático na quantização
    return torch.round(tensor.double() * SCALING_FACTOR).to(torch.int64)


def fixed_point_to_float(tensor):
    return (tensor.double() / SCALING_FACTOR).to(torch.float32)


def gerar_shares_smpc_modulares(state_dict, num_shares, gerador):
    shares = [{} for _ in range(num_shares)]
    for key, tensor in state_dict.items():
        tensor_int = torch.remainder(float_to_fixed_point(tensor.cpu()), MODULUS)
        aleatorios = [torch.randint(0, MODULUS, tensor_int.shape, dtype=torch.int64, generator=gerador)
                      for _ in range(num_shares - 1)]
        ultimo = tensor_int
        for r in aleatorios:
            ultimo = torch.remainder(ultimo - r, MODULUS)
        for i in range(num_shares - 1):
            shares[i][key] = aleatorios[i]
        shares[-1][key] = ultimo
    return shares


def agregar_smpc(estados_D, semente):
    """Cada cliente divide seus pesos em shares; cada nó soma às cegas; o servidor só vê a soma total."""
    n = len(estados_D)
    # Limite do campo: a soma dos n clientes precisa caber em (-p/2, p/2) no ponto fixo
    limite = (MODULUS // 2) / SCALING_FACTOR
    soma_abs = sum(max(t.abs().max().item() for t in e.values()) for e in estados_D)
    assert soma_abs < limite, f"Pesos grandes demais para o campo SMPC ({soma_abs:.1f} >= {limite:.1f})"

    gerador = torch.Generator().manual_seed(semente + 777)  # Em produção: CSPRNG
    sincronizar()
    t0 = time.perf_counter()
    canal = {j: [] for j in range(n)}
    for estado in estados_D:
        for j, share in enumerate(gerar_shares_smpc_modulares(estado, n, gerador)):
            canal[j].append(share)

    parciais = []
    for j in range(n):
        parciais.append({k: torch.remainder(sum(s[k] for s in canal[j]), MODULUS) for k in estados_D[0]})

    agregado = {}
    for k in estados_D[0]:
        total = torch.remainder(sum(p[k] for p in parciais), MODULUS)
        total = torch.where(total > MODULUS // 2, total - MODULUS, total)
        agregado[k] = (fixed_point_to_float(total) / n).to(device)
    return agregado, time.perf_counter() - t0


def agregar_claro(estados_D):
    n = len(estados_D)
    return {k: sum(e[k].double() for e in estados_D).float() / n for k in estados_D[0]}

In [ ]:
# =====================================================================
# --- 4. AVALIAÇÃO ---
# =====================================================================

def scores_anomalia(estado_D, input_dim, X, tamanho_lote=65536):
    D = Discriminator(input_dim).to(device)
    D.load_state_dict(estado_D)
    D.eval()
    saida = []
    with torch.no_grad():
        for i in range(0, len(X), tamanho_lote):
            saida.append(D(torch.from_numpy(X[i:i + tamanho_lote]).to(device)).cpu().numpy().ravel())
    # D -> ~0.9 para normal e ~0.1 para fraude: 1 - D(x) é o escore de anomalia
    return 1.0 - np.concatenate(saida)


def escolher_limiar(y_val, s_val):
    """Estratégia do Teste 4, aplicada SÓ na validação: maior recall com precisão >= PRECISAO_MINIMA."""
    precisions, recalls, thresholds = precision_recall_curve(y_val, s_val)
    validos = np.where(precisions[:-1] >= PRECISAO_MINIMA)[0]
    if len(validos) > 0:
        return thresholds[validos[np.argmax(recalls[validos])]]
    f1 = np.divide(2 * precisions * recalls, precisions + recalls,
                   out=np.zeros_like(precisions), where=(precisions + recalls) != 0)
    return thresholds[np.argmax(f1[:-1])]


def avaliar(estado_D, dados):
    limiar = escolher_limiar(dados['y_val'], scores_anomalia(estado_D, dados['input_dim'], dados['X_val']))
    s = scores_anomalia(estado_D, dados['input_dim'], dados['X_test'])
    y, y_pred = dados['y_test'], (s >= limiar).astype(int)
    return dict(scores=s, limiar=limiar, **{
        'ROC-AUC': roc_auc_score(y, s), 'PR-AUC': average_precision_score(y, s),
        'Precisão': precision_score(y, y_pred, zero_division=0), 'Recall': recall_score(y, y_pred, zero_division=0),
        'F1': f1_score(y, y_pred, zero_division=0), 'Acurácia': accuracy_score(y, y_pred)})


def bootstrap_delta(y, pares_scores, n_boot, semente):
    """
    IC 95% do Δ médio entre sementes, métrica(federado) − métrica(centralizado), reamostrando o teste.
    pares_scores: lista de (scores_centralizado, scores_federado), uma por semente, avaliados nos mesmos índices.
    """
    rng = np.random.default_rng(semente)
    deltas = {'ROC-AUC': [], 'PR-AUC': []}
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        if y[idx].min() == y[idx].max():
            continue
        for m, fn in [('ROC-AUC', roc_auc_score), ('PR-AUC', average_precision_score)]:
            deltas[m].append(np.mean([fn(y[idx], b[idx]) - fn(y[idx], a[idx]) for a, b in pares_scores]))
    return {m: np.array(v) for m, v in deltas.items()}

In [ ]:
# =====================================================================
# --- 5. EXPERIMENTO: CENTRALIZADO × FEDERADO + SMPC ---
# =====================================================================

METRICAS = ['ROC-AUC', 'PR-AUC', 'Precisão', 'Recall', 'F1', 'Acurácia']
CONFIGS = ['Centralizado', 'Federado + SMPC', 'Federado + FedAvg em claro', 'Local isolado']
CORES = {'Centralizado': '#1f77b4', 'Federado + SMPC': '#d62728',
         'Federado + FedAvg em claro': '#ff7f0e', 'Local isolado': '#7f7f7f'}
resultados_gerais = {}

for nome_base in DATASETS:
    print("\n" + "=" * 78)
    print(f" BASE: {nome_base.upper()}")
    print("=" * 78)
    dados = carregar_dados(nome_base)
    if dados is None:
        print(f"Aviso: base '{nome_base}' não encontrada em {CAMINHOS[nome_base]}. Pulando.")
        continue
    input_dim = dados['input_dim']

    linhas, tempos, pares_scores, scores_seed0 = [], [], [], {}
    for semente in SEMENTES:
        print(f"\n--- Semente {semente} ---")
        # Pesos iniciais idênticos para o centralizado e para todos os clientes
        seed_everything(semente)
        G0 = copy.deepcopy(Generator(input_dim).state_dict())
        D0 = copy.deepcopy(Discriminator(input_dim).state_dict())

        # (a) Centralizado
        D_central, t_central = treinar_gan(G0, D0, dados['X_central'], dados['y_central'], input_dim, semente * 100)
        print(f"[Centralizado] treino: {t_central:.1f}s")

        # (b) Federado: treino local em cada silo
        estados_locais, t_clientes = [], []
        for k, (X_k, y_k) in enumerate(dados['clientes']):
            D_k, t_k = treinar_gan(G0, D0, X_k, y_k, input_dim, semente * 100 + k + 1)
            estados_locais.append(D_k)
            t_clientes.append(t_k)
        print(f"[Federado] treino local: {[f'{t:.1f}s' for t in t_clientes]}")

        D_smpc, t_smpc = agregar_smpc(estados_locais, semente)
        D_claro = agregar_claro(estados_locais)
        erro_smpc = max((D_smpc[k].cpu() - D_claro[k].cpu()).abs().max().item() for k in D_claro)
        print(f"[SMPC] agregação: {1000 * t_smpc:.1f} ms | erro máx. vs FedAvg em claro: {erro_smpc:.2e}")

        avaliacoes = {'Centralizado': avaliar(D_central, dados),
                      'Federado + SMPC': avaliar(D_smpc, dados),
                      'Federado + FedAvg em claro': avaliar(D_claro, dados)}
        locais = [avaliar(e, dados) for e in estados_locais]
        avaliacoes['Local isolado'] = {m: np.mean([a[m] for a in locais]) for m in METRICAS}

        for cfg, av in avaliacoes.items():
            linhas.append({'Semente': semente, 'Configuração': cfg, **{m: av[m] for m in METRICAS}})
            print(f"  {cfg:28s} ROC-AUC {av['ROC-AUC']:.4f} | PR-AUC {av['PR-AUC']:.4f} | F1 {av['F1']:.4f}")
        tempos.append({'Semente': semente, 'Centralizado': t_central, 'Federado (sequencial)': sum(t_clientes),
                       'Federado (paralelo)': max(t_clientes), 'Agregação SMPC': t_smpc})
        pares_scores.append((avaliacoes['Centralizado']['scores'], avaliacoes['Federado + SMPC']['scores']))
        if semente == SEMENTES[0]:
            scores_seed0.update({cfg: avaliacoes[cfg]['scores'] for cfg in CONFIGS[:3]})

    df = pd.DataFrame(linhas)
    df_tempos = pd.DataFrame(tempos)

    # ----- Resumo: média ± desvio por configuração -----
    resumo = df.groupby('Configuração')[METRICAS].agg(['mean', 'std']).reindex(CONFIGS)
    print(f"\n>> MÉTRICAS NO TESTE: média ± desvio em {len(SEMENTES)} sementes (limiar escolhido na validação)")
    tabela = pd.DataFrame({m: [f"{resumo.loc[c, (m, 'mean')]:.4f} ± {np.nan_to_num(resumo.loc[c, (m, 'std')]):.4f}"
                               for c in CONFIGS] for m in METRICAS}, index=CONFIGS)
    print(tabela.to_string())

    # ----- Deltas pareados por semente (Federado + SMPC − Centralizado) -----
    piv = df.pivot(index='Semente', columns='Configuração')
    print("\n>> Δ PAREADO POR SEMENTE (Federado + SMPC − Centralizado):")
    deltas_seed = {}
    for m in METRICAS:
        d = (piv[(m, 'Federado + SMPC')] - piv[(m, 'Centralizado')]).values
        deltas_seed[m] = d
        if len(d) > 1:
            meia = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d))
            print(f"  {m:9s}: {d.mean():+.4f}  (IC95% entre sementes: [{d.mean() - meia:+.4f}, {d.mean() + meia:+.4f}])")
        else:
            print(f"  {m:9s}: {d.mean():+.4f}")

    # ----- Bootstrap pareado no teste (Δ médio entre sementes) -----
    boot = bootstrap_delta(dados['y_test'], pares_scores, N_BOOTSTRAP, SEMENTES[0])
    ic = {m: np.percentile(v, [2.5, 97.5]) for m, v in boot.items()}
    print(f"\n>> IC 95% POR BOOTSTRAP ({N_BOOTSTRAP} reamostragens do teste, Δ médio entre sementes):")
    for m in ['ROC-AUC', 'PR-AUC']:
        print(f"  Δ{m}: [{ic[m][0]:+.4f}, {ic[m][1]:+.4f}]")

    delta_roc = deltas_seed['ROC-AUC'].mean()
    nao_inferior = ic['ROC-AUC'][0] > -MARGEM_NAO_INFERIORIDADE
    print("\n>> RESPOSTA À QUESTÃO DE PESQUISA:")
    if nao_inferior:
        print(f"  O STEP-GAN Federado + SMPC é NÃO-INFERIOR ao baseline centralizado (margem {MARGEM_NAO_INFERIORIDADE}):"
              f" ΔROC-AUC médio {delta_roc:+.4f}, limite inferior do IC95% {ic['ROC-AUC'][0]:+.4f} > "
              f"{-MARGEM_NAO_INFERIORIDADE}.")
    else:
        print(f"  NÃO foi possível demonstrar não-inferioridade (margem {MARGEM_NAO_INFERIORIDADE}):"
              f" ΔROC-AUC médio {delta_roc:+.4f}, limite inferior do IC95% {ic['ROC-AUC'][0]:+.4f} <= "
              f"{-MARGEM_NAO_INFERIORIDADE}.")
    custo_smpc = (piv[('ROC-AUC', 'Federado + SMPC')] - piv[('ROC-AUC', 'Federado + FedAvg em claro')]).abs().max()
    ganho_colab = (piv[('ROC-AUC', 'Federado + SMPC')] - piv[('ROC-AUC', 'Local isolado')]).mean()
    print(f"  Efeito do SMPC isolado (|Δ ROC-AUC| SMPC vs FedAvg em claro): {custo_smpc:.2e}")
    print(f"  Ganho da colaboração (Federado + SMPC − Local isolado): {ganho_colab:+.4f} ROC-AUC")

    print("\n>> TEMPO DE TREINO (s, média entre sementes):")
    print(df_tempos.drop(columns='Semente').mean().to_frame('segundos').T.to_string(float_format=lambda x: f"{x:.3f}"))
    resultados_gerais[nome_base] = dict(metricas=df, tempos=df_tempos, ic_bootstrap=ic, nao_inferior=nao_inferior)

    # ----- Visualizações -----
    fig, axes = plt.subplots(2, 2, figsize=(18, 11))
    fig.suptitle(f"Teste 6: Centralizado × Federado + SMPC (STEP-GAN, {nome_base})", fontsize=14, fontweight='bold')

    # 1. Métricas (média ± desvio)
    ax = axes[0, 0]
    largura = 0.2
    x = np.arange(len(METRICAS))
    for i, cfg in enumerate(CONFIGS):
        ax.bar(x + (i - 1.5) * largura, [resumo.loc[cfg, (m, 'mean')] for m in METRICAS], largura,
               yerr=[np.nan_to_num(resumo.loc[cfg, (m, 'std')]) for m in METRICAS], capsize=3,
               color=CORES[cfg], label=cfg)
    ax.set_xticks(x)
    ax.set_xticklabels(METRICAS)
    ax.set_ylim(0, 1.2)
    ax.set_title(f'Métricas no teste (média ± desvio, {len(SEMENTES)} sementes)')
    ax.legend(loc='upper center', ncol=2)

    # 2. Curvas ROC (primeira semente)
    ax = axes[0, 1]
    for cfg in CONFIGS[:3]:
        fpr, tpr, _ = roc_curve(dados['y_test'], scores_seed0[cfg])
        ax.plot(fpr, tpr, '-' if cfg == 'Centralizado' else '--', color=CORES[cfg],
                label=f"{cfg} (AUC={roc_auc_score(dados['y_test'], scores_seed0[cfg]):.4f})")
    ax.plot([0, 1], [0, 1], ':', color='gray')
    ax.set_title(f'Curva ROC no teste (semente {SEMENTES[0]})')
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.legend(loc='lower right')

    # 3. Δ por semente e IC de não-inferioridade
    ax = axes[1, 0]
    for i, m in enumerate(['ROC-AUC', 'PR-AUC', 'F1']):
        ax.scatter(np.full(len(deltas_seed[m]), i), deltas_seed[m], color=CORES['Federado + SMPC'], zorder=3,
                   label='Δ por semente' if i == 0 else None)
        ax.plot([i - 0.2, i + 0.2], [deltas_seed[m].mean()] * 2, color='black', lw=2,
                label='Δ médio' if i == 0 else None)
    for i, m in enumerate(['ROC-AUC', 'PR-AUC']):
        ax.errorbar(i + 0.3, np.mean(ic[m]), yerr=[[np.mean(ic[m]) - ic[m][0]], [ic[m][1] - np.mean(ic[m])]],
                    fmt='none', ecolor='#2ca02c', capsize=6, lw=2, label='IC95% bootstrap' if i == 0 else None)
    ax.axhline(0, color='gray', lw=1)
    ax.axhline(-MARGEM_NAO_INFERIORIDADE, color='#d62728', ls='--', label=f'Margem de não-inferioridade (−{MARGEM_NAO_INFERIORIDADE})')
    ax.set_xticks(range(3))
    ax.set_xticklabels(['ΔROC-AUC', 'ΔPR-AUC', 'ΔF1'])
    ax.set_title('Federado + SMPC − Centralizado')
    ax.legend(loc='best', fontsize=9)

    # 4. Tempo de treino
    ax = axes[1, 1]
    media_t = df_tempos[['Centralizado', 'Federado (sequencial)', 'Federado (paralelo)']].mean()
    ax.bar(media_t.index, media_t.values, color=['#1f77b4', '#ff9896', '#d62728'])
    for i, v in enumerate(media_t.values):
        ax.text(i, v, f"{v:.1f}s", ha='center', va='bottom')
    ax.set_title(f"Tempo de treino (agregação SMPC: {1000 * df_tempos['Agregação SMPC'].mean():.1f} ms)")
    ax.set_ylabel('segundos')

    plt.tight_layout()
    plt.show()

print("\n[SUCESSO] Teste 6 concluído!")